# SnowHealth Demand Forecasting - Hyperparameter Tuning
Compare baseline XGBoost vs hyperparameter-tuned XGBoost using RandomizedSearchCV.
Also demonstrates the Snowflake ML Tuner API pattern for distributed HPO on Container Runtime.

In [ ]:
# Initialize
import sys, os
work_dir = os.getcwd()
sys.path.insert(0, os.path.join(work_dir, 'source'))

from snowflake.snowpark.context import get_active_session
import config as cfg
from model_utils import compute_metrics, get_feature_importance, save_feature_importance_artifact, generate_comparison_report
from feature_engineering import get_feature_columns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_MODELS_SCHEMA}").collect()
print("Session ready")

In [ ]:
# Load features and prepare data
df = session.table(f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.SNOWHEALTH_DEMAND_FEATURES").to_pandas()
feature_cols = get_feature_columns(df, cfg.TARGET_COLUMN)

X = df[feature_cols].astype(float)
y = df[cfg.TARGET_COLUMN].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=cfg.TEST_SIZE, random_state=cfg.RANDOM_STATE)
print(f"Data ready: {X_train.shape[0]} train, {X_test.shape[0]} test, {len(feature_cols)} features")

In [ ]:
# ===== BASELINE MODEL =====
from xgboost import XGBRegressor

xgb_baseline = XGBRegressor(**cfg.XGBOOST_BASELINE_PARAMS)
xgb_baseline.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

baseline_preds = xgb_baseline.predict(X_test)
baseline_metrics = compute_metrics(y_test.values, baseline_preds)
print("Baseline XGBoost Results:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# ===== HYPERPARAMETER TUNING WITH RANDOMIZEDSEARCHCV =====
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'reg_alpha': [0, 0.01, 0.1, 1.0],
    'reg_lambda': [0.5, 1.0, 2.0, 5.0],
}

xgb_tuner = XGBRegressor(objective='reg:squarederror', tree_method='hist', random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_tuner,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    verbose=1,
    n_jobs=-1,
)

print("Starting hyperparameter search (20 configurations, 3-fold CV)...")
random_search.fit(X_train, y_train)
print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best CV RMSE: {-random_search.best_score_:.4f}")

In [ ]:
# Evaluate tuned model on test set
tuned_model = random_search.best_estimator_
tuned_preds = tuned_model.predict(X_test)
tuned_metrics = compute_metrics(y_test.values, tuned_preds)

print("Tuned XGBoost Results:")
for k, v in tuned_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# ===== COMPARISON: BASELINE vs TUNED =====
comparison = generate_comparison_report(baseline_metrics, tuned_metrics)
print("\n" + "=" * 70)
print("BASELINE vs TUNED COMPARISON")
print("=" * 70)
print(comparison.to_string(index=False))
print("=" * 70)

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metric_names = ['MAE', 'RMSE', 'MAPE', 'R2']
baseline_vals = [baseline_metrics['mae'], baseline_metrics['rmse'], baseline_metrics['mape'], baseline_metrics['r2']]
tuned_vals = [tuned_metrics['mae'], tuned_metrics['rmse'], tuned_metrics['mape'], tuned_metrics['r2']]

x = np.arange(len(metric_names))
width = 0.35
axes[0].bar(x - width/2, baseline_vals, width, label='Baseline', color='#999999')
axes[0].bar(x + width/2, tuned_vals, width, label='Tuned', color='#cc0000')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names)
axes[0].legend()
axes[0].set_title('Metrics Comparison', fontsize=13, fontweight='bold')

# Residual distribution
baseline_residuals = y_test.values - baseline_preds
tuned_residuals = y_test.values - tuned_preds
axes[1].hist(baseline_residuals, bins=40, alpha=0.5, color='#999999', label='Baseline', edgecolor='white')
axes[1].hist(tuned_residuals, bins=40, alpha=0.5, color='#cc0000', label='Tuned', edgecolor='white')
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].legend()

plt.suptitle('Baseline vs Tuned XGBoost', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Log tuned model experiment and save feature importance
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
exp.set_experiment(cfg.SNOWHEALTH_EXPERIMENT_NAME)

with exp.start_run("xgboost_tuned_hpo"):
    exp.log_params(random_search.best_params_)
    exp.log_param("model_type", "XGBRegressor_Tuned")
    exp.log_param("hpo_method", "RandomizedSearchCV")
    exp.log_param("n_iter", 20)
    exp.log_param("cv_folds", 3)
    exp.log_metrics(tuned_metrics)
    exp.log_metric("cv_best_rmse", -random_search.best_score_)
    
    # Feature importance artifact
    tuned_fi = get_feature_importance(tuned_model, feature_cols, "xgboost")
    fi_path = save_feature_importance_artifact(tuned_fi, "xgboost_tuned")
    exp.log_artifact(fi_path, artifact_path="feature_importance")

print("Tuned model experiment logged")

In [ ]:
# Register tuned model to Model Registry
from snowflake.ml.registry import Registry
from snowflake.ml.model.model_signature import infer_signature

registry = Registry(session=session, database_name=cfg.SNOWHEALTH_DATABASE, schema_name=cfg.SNOWHEALTH_MODELS_SCHEMA)
sig = infer_signature(X_train.head(10), y_train.head(10))

mv_tuned = registry.log_model(
    tuned_model,
    model_name=cfg.SNOWHEALTH_MODEL_NAME,
    version_name="tuned_v1",
    signatures={"predict": sig},
    comment=f"SNOWHEALTH demand forecast - HPO tuned XGBoost (RMSE: {tuned_metrics['rmse']})",
    metrics=tuned_metrics,
    sample_input_data=X_train.head(10),
    target_platforms=["WAREHOUSE"],
)
print(f"Tuned model registered: {cfg.SNOWHEALTH_MODEL_NAME} v tuned_v1")

# List all versions
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
for v in model_ref.versions():
    print(f"  {v.version_name}: {v.comment}")

In [ ]:
# Set tuned model as default
model_ref = registry.get_model(cfg.SNOWHEALTH_MODEL_NAME)
model_ref.default = "tuned_v1"
print(f"Default version set to: tuned_v1")

print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING COMPLETE")
print("=" * 60)
print(f"Baseline RMSE:  {baseline_metrics['rmse']}")
print(f"Tuned RMSE:     {tuned_metrics['rmse']}")
improvement = baseline_metrics['rmse'] - tuned_metrics['rmse']
print(f"Improvement:    {improvement:.4f} ({improvement/baseline_metrics['rmse']*100:.1f}%)")
print(f"Best params:    {random_search.best_params_}")
print("=" * 60)
print("Proceed to 05_deployment_monitoring.ipynb")

## Snowflake ML Tuner API (Container Runtime)
The code below shows the Snowflake-native Tuner API for distributed HPO.
This requires Container Runtime (SPCS compute pools) and is included as a reference pattern.

```python
from snowflake.ml.modeling.tune import Tuner, TunerConfig, get_tuner_context, uniform, loguniform, randint
from snowflake.ml.modeling.tune.search import RandomSearch
from snowflake.ml.data.data_connector import DataConnector

def train_func():
    ctx = get_tuner_context()
    params = ctx.get_hyper_params()
    datasets = ctx.get_dataset_map()
    
    train_df = datasets['train'].to_pandas()
    test_df = datasets['test'].to_pandas()
    
    X_tr = train_df.drop('UNITS_SOLD', axis=1)
    y_tr = train_df['UNITS_SOLD']
    X_te = test_df.drop('UNITS_SOLD', axis=1)
    y_te = test_df['UNITS_SOLD']
    
    model = XGBRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        subsample=params['subsample'],
        objective='reg:squarederror',
    )
    model.fit(X_tr, y_tr)
    rmse = np.sqrt(mean_squared_error(y_te, model.predict(X_te)))
    ctx.report(metrics={'rmse': rmse}, model=model)

search_space = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(4, 12),
    'learning_rate': loguniform(0.01, 0.3),
    'subsample': uniform(0.7, 1.0),
}

config = TunerConfig(metric='rmse', mode='min', search_alg=RandomSearch(), num_trials=20)
tuner = Tuner(train_func, search_space, config)
results = tuner.run(dataset_map={
    'train': DataConnector.from_dataframe(session.table('SNOWHEALTH_ML_DB.SNOWHEALTH_FEATURES.SNOWHEALTH_TRAIN_DATA')),
    'test': DataConnector.from_dataframe(session.table('SNOWHEALTH_ML_DB.SNOWHEALTH_FEATURES.SNOWHEALTH_TEST_DATA')),
})

best_model = results.best_model
print(f"Best RMSE: {results.best_result['rmse'].iloc[0]}")
```